In [7]:
from google.colab import drive
drive.mount('/content/drive')

# Set your working path
data_path = "/content/preprocessed_news.csv"
output_path = "/content/news_with_bias_and_fake_labels.csv"



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install transformers pandas tqdm


In [4]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

# Load classifier (runs on GPU if available)
zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

# Define labels
labels = ["left-wing", "right-wing", "neutral", "real", "fake"]
bias_set = {"left-wing", "right-wing", "neutral"}
truth_set = {"real", "fake"}


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


In [5]:
def detect_bias_and_truth(text):
    try:
        result = zero_shot_classifier(text, candidate_labels=labels)
        scores = dict(zip(result["labels"], result["scores"]))

        truth_label = max(truth_set, key=lambda l: scores.get(l, 0))
        truth_confidence = scores[truth_label]

        bias_label = max(bias_set, key=lambda l: scores.get(l, 0))
        bias_confidence = scores[bias_label]

        return truth_label, truth_confidence, bias_label, bias_confidence
    except:
        return "error", 0.0, "error", 0.0


In [8]:
df = pd.read_csv(data_path)
tqdm.pandas()

# Use only first 512 characters for speed (optional)
df["text_for_analysis"] = df["translated_text"].astype(str).str.slice(0, 512)

# Run classification
df[["truth_prediction", "truth_confidence", "bias_label", "bias_confidence"]] = df["text_for_analysis"].progress_apply(
    lambda x: pd.Series(detect_bias_and_truth(x))
)

# Save output
df.to_csv(output_path, index=False)
print("✅ Done! Output saved to:", output_path)


100%|██████████| 917/917 [01:54<00:00,  7.98it/s]

✅ Done! Output saved to: /content/news_with_bias_and_fake_labels.csv


In [9]:
from google.colab import files
files.download("news_with_bias_and_fake_labels.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>